In [ ]:
from langchain_neo4j import Neo4jGraph

NEO4J_URI="bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""
NEO4J_DATABASE = "neo4j"

kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

## Demo Knowledge Graph basic

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)


In [4]:
text_example = """
Theo Bộ luật Lao động Việt Nam, người lao động và người sử dụng lao động
có thể thiết lập quan hệ lao động thông qua hợp đồng lao động. Hợp đồng
lao động quy định công việc, tiền lương, thời giờ làm việc, thời giờ nghỉ
ngơi và quyền, nghĩa vụ của các bên.

Người sử dụng lao động có nghĩa vụ trả lương đầy đủ và đúng hạn cho người
lao động, đồng thời phải bảo đảm điều kiện an toàn, vệ sinh lao động tại
nơi làm việc. Thời giờ làm việc bình thường của người lao động không được
vượt quá 8 giờ trong một ngày và 48 giờ trong một tuần.

Khi người lao động làm thêm giờ vào ngày làm việc bình thường, họ được trả
ít nhất 150% mức tiền lương; nếu làm thêm vào ngày nghỉ hằng tuần, mức trả
ít nhất là 200%; còn khi làm thêm vào ngày lễ, Tết hoặc ngày nghỉ có hưởng
lương, mức trả ít nhất là 300%.

Người lao động cũng có quyền được nghỉ hằng năm và tham gia bảo hiểm xã hội,
bảo hiểm y tế và bảo hiểm thất nghiệp theo quy định của pháp luật.

Trong quá trình làm việc, nếu người lao động vi phạm nội quy lao động,
người sử dụng lao động có thể áp dụng các hình thức kỷ luật lao động như
khiển trách, kéo dài thời hạn nâng lương, cách chức hoặc sa thải theo các
điều kiện luật định.

Khi phát sinh tranh chấp lao động giữa người lao động và người sử dụng lao
động, tranh chấp có thể được giải quyết thông qua hòa giải viên lao động,
Hội đồng trọng tài lao động hoặc Tòa án tùy thuộc vào loại tranh chấp và
trường hợp cụ thể.
""".strip()

In [ ]:
import json 

def extract_entities_and_relationships(text):
    
    from openai import OpenAI
    client = OpenAI(
        # api_key = os.getenv("OPENAI_API_KEY")
        api_key = os.getenv("OPENAI_API_KEY")
    )
    
    prompt = f"""
                You are an information extraction system that builds a knowledge graph from Vietnamese legal or administrative text.

                Your task is to extract entities (nodes) and relationships (edges) that are explicitly mentioned in the text.

                Important rules:
                - Extract only information that is directly stated in the text.
                - Do not infer, guess, summarize, or add external knowledge.
                - Keep entity names in Vietnamese exactly as they appear in the text when possible.
                - Relationship types must be concise, normalized, and written in Vietnamese.
                - Entity types must be concise and written in Vietnamese.
                - Merge duplicate entities that refer to the same real-world object.
                - Do not extract vague or generic words as entities unless they refer to a specific concept, organization, document, person, place, regulation, or event.
                - If there are no valid entities or relationships, return empty lists.
                - Return valid JSON only. Do not include markdown, explanations, comments, or extra text.

                Entity extraction guidelines:
                Each entity must have:
                - "name": the entity name
                - "type": the entity category

                Suggested entity types:
                - "Văn bản pháp luật"
                - "Cơ quan"
                - "Tổ chức"
                - "Cá nhân"
                - "Địa điểm"
                - "Thời gian"
                - "Khái niệm"
                - "Quy định"
                - "Nghĩa vụ"
                - "Quyền lợi"
                - "Hành vi"
                - "Đối tượng áp dụng"
                - "Mức phạt"
                - "Điều khoản"
                - ... (tự nghĩ nếu bạn tìm được type phù hợp hơn)

                Relationship extraction guidelines:
                Each relationship must have:
                - "source": the source entity name
                - "type": the relationship type
                - "target": the target entity name

                Suggested relationship types:
                - "ban hành"
                - "quy định"
                - "áp dụng cho"
                - "có hiệu lực từ"
                - "sửa đổi"
                - "bổ sung"
                - "thay thế"
                - "liên quan đến"
                - "có nghĩa vụ"
                - "có quyền"
                - "bị xử phạt bởi"
                - "thuộc"
                - "gồm"
                - "căn cứ vào"
                - ... (nếu bạn tìm được relationships phù hợp hơn)

                Output schema:
                {{
                "entities": [
                    {{
                    "name": "...",
                    "type": "..."
                    }}
                ],
                "relationships": [
                    {{
                    "source": "...",
                    "type": "...",
                    "target": "..."
                    }}
                ]
                }}

                Text:
                {text}
            """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a precise knowledge graph extraction engine for Vietnamese legal "
                            "and administrative documents. Extract only explicitly stated entities and "
                            "relationships from the provided text. Do not infer, guess, summarize, or "
                            "add external knowledge. Return valid JSON only, following the requested schema."
                        ),
                    },
                    {"role": "user", "content": prompt},
                ],
        temperature=0,
        response_format={"type": "json_object"},
    )

    return json.loads(response.choices[0].message.content)

In [7]:
import pprint
result = extract_entities_and_relationships(text_example)
pprint.pprint(result)

{'entities': [{'name': 'Bộ luật Lao động Việt Nam',
               'type': 'Văn bản pháp luật'},
              {'name': 'người lao động', 'type': 'Đối tượng áp dụng'},
              {'name': 'người sử dụng lao động', 'type': 'Đối tượng áp dụng'},
              {'name': 'hợp đồng lao động', 'type': 'Khái niệm'},
              {'name': 'công việc', 'type': 'Quy định'},
              {'name': 'tiền lương', 'type': 'Quy định'},
              {'name': 'thời giờ làm việc', 'type': 'Quy định'},
              {'name': 'thời giờ nghỉ ngơi', 'type': 'Quy định'},
              {'name': 'quyền', 'type': 'Quyền lợi'},
              {'name': 'nghĩa vụ', 'type': 'Nghĩa vụ'},
              {'name': 'an toàn lao động', 'type': 'Quy định'},
              {'name': 'vệ sinh lao động', 'type': 'Quy định'},
              {'name': 'thời giờ làm việc bình thường', 'type': 'Quy định'},
              {'name': 'làm thêm giờ', 'type': 'Hành vi'},
              {'name': 'ngày làm việc bình thường', 'type': 'Thời g

In [8]:
import json

with open("output.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

In [12]:
print("==========Entities==========")
for entity in result['entities']:
    print(f'{entity['name']}: {entity['type']}')

relationship_list = []
print("\n\n==========Relationships==========")
for relationship in result['relationships']:
    print(f'{relationship['source']} --> {relationship['type']} --> {relationship['target']}')
    relationship_list.append((relationship['source'], relationship['type'], relationship['target']))

==========Entities==========
Bộ luật Lao động Việt Nam: Văn bản pháp luật
người lao động: Đối tượng áp dụng
người sử dụng lao động: Đối tượng áp dụng
hợp đồng lao động: Khái niệm
công việc: Quy định
tiền lương: Quy định
thời giờ làm việc: Quy định
thời giờ nghỉ ngơi: Quy định
quyền: Quyền lợi
nghĩa vụ: Nghĩa vụ
an toàn lao động: Quy định
vệ sinh lao động: Quy định
thời giờ làm việc bình thường: Quy định
làm thêm giờ: Hành vi
ngày làm việc bình thường: Thời gian
ngày nghỉ hằng tuần: Thời gian
ngày lễ, Tết: Thời gian
ngày nghỉ có hưởng lương: Thời gian
nghỉ hằng năm: Quyền lợi
bảo hiểm xã hội: Quyền lợi
bảo hiểm y tế: Quyền lợi
bảo hiểm thất nghiệp: Quyền lợi
nội quy lao động: Quy định
kỷ luật lao động: Hành vi
khiển trách: Hành vi
kéo dài thời hạn nâng lương: Hành vi
cách chức: Hành vi
sa thải: Hành vi
tranh chấp lao động: Khái niệm
hòa giải viên lao động: Cá nhân
Hội đồng trọng tài lao động: Tổ chức
Tòa án: Cơ quan


==========Relationships==========
Bộ luật Lao động Việt Nam --> quy đ

In [15]:
print(len(relationship_list))

28


In [16]:
def add_relationships_to_neo4j(kg, relationships):
    """
    Add relationships extracted from the knowledge graph to the Neo4j database.
    """
    with kg._driver.session() as session:
        for subject, relation, obj in relationships:
            # Create nodes and relationships in Neo4j

            cypher_query = f"""
            MERGE (a:Entity {{name: $subject}})
            MERGE (b:Entity {{name: $object}})
            MERGE (a)-[:`{relation}`]->(b)
            """
            session.run(cypher_query, subject=subject, object=obj)
    print("Relationships added to Neo4j.")

In [17]:
add_relationships_to_neo4j(kg, relationship_list)

Relationships added to Neo4j.


In [18]:
def visualize_graph(data):
    try:
        import networkx as nx
        from pyvis.network import Network
        from IPython.display import IFrame, display

        # Create a directed graph
        G = nx.DiGraph()

        # Process the data to populate the graph
        for record in data:
            # Extract node_a, node_b, and the relationship type
            node_a = record.get('node_a')
            node_b = record.get('node_b')
            relationship = record.get('r')[1] if 'r' in record and isinstance(record['r'], tuple) else None

            if node_a and node_b and relationship:
                # Add nodes and edge with the relationship type as a label
                G.add_node(node_a, label=node_a)
                G.add_node(node_b, label=node_b)
                G.add_edge(node_a, node_b, label=relationship)

        # Create a Pyvis Network object with CDN resources set to 'in_line'
        net = Network(notebook=True, directed=True)
        net.from_nx(G)

        # Add edge labels for better visualization
        for edge in G.edges(data=True):
            net.add_edge(edge[0], edge[1], title=edge[2].get('label', ''))

        # Save the graph to an HTML file
        net.show('graph.html')

        # Embed the graph in Google Colab using IFrame
        display(IFrame('graph.html', width='100%', height='100%'))

    except Exception as e:
        print(f"Error while visualizing the graph: {str(e)}")
        return None


In [19]:
from pyvis.network import Network
from IPython.display import IFrame, display

rows = kg.query("""
MATCH (a)-[r]->(b)
RETURN
  elementId(a) AS source_id,
  labels(a) AS source_labels,
  coalesce(a.name, a.title, a.id, elementId(a)) AS source,
  type(r) AS rel,
  elementId(b) AS target_id,
  labels(b) AS target_labels,
  coalesce(b.name, b.title, b.id, elementId(b)) AS target
LIMIT 300
""")

net = Network(
    height="750px",
    width="100%",
    directed=True,
    notebook=True,
    cdn_resources="in_line",
)

net.barnes_hut(
    gravity=-30000,
    central_gravity=0.3,
    spring_length=160,
    spring_strength=0.04,
    damping=0.09,
)

for row in rows:
    source_group = row["source_labels"][0] if row["source_labels"] else "Entity"
    target_group = row["target_labels"][0] if row["target_labels"] else "Entity"

    net.add_node(
        row["source_id"],
        label=row["source"],
        title=f"{row['source']} ({source_group})",
        group=source_group,
    )

    net.add_node(
        row["target_id"],
        label=row["target"],
        title=f"{row['target']} ({target_group})",
        group=target_group,
    )

    net.add_edge(
        row["source_id"],
        row["target_id"],
        label=row["rel"],
        title=row["rel"],
        arrows="to",
    )

output_path = "kg_visualization.html"
net.write_html(output_path)

display(IFrame(output_path, width="100%", height=760))

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `title` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=22, offset=107>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 107, 'line': 6, 'column': 22}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (a)-[r]->(b)\nRETURN\n  elementId(a) AS source_id,\n  labels(a) AS source_labels,\n  coalesce(a.name, a.title, a.id, elementId(a)) AS source,\n  type(r) AS rel,\n  elementId(b) AS target_id,\n  labels(b) AS target_labels,\n  coalesce(b.name, b.title, b.id, elementId(b)) AS target\nLIMIT 300\n'
Received not

## Xây dựng Knowledge Graph với văn bản Luật Lao động

### Pipeline config


In [22]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import time


# ===== Config =====
# Works whether the notebook is launched from project root or from chatbot/.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists() and (PROJECT_DIR / "chatbot" / "data").exists():
    PROJECT_DIR = PROJECT_DIR / "chatbot"

DATA_DIR = PROJECT_DIR / "data"
CACHE_PATH = PROJECT_DIR / "kg_extraction_cache.json"
GRAPH_EXPORT_PATH = PROJECT_DIR / "kg_graph_all_data.json"
VISUALIZATION_PATH = PROJECT_DIR / "kg_visualization_all_data.html"

# Set a small integer while testing, then set to None for the full dataset.
MAX_ARTICLES = None
SLEEP_SECONDS_BETWEEN_CALLS = 0.2

# Keep False by default so you do not accidentally delete previous graph data.
# This only deletes nodes created by this pipeline: KGDocument, KGArticle, KGEntity.
RESET_PIPELINE_GRAPH = False


### Read and split markdown data


In [23]:
# ===== Read and split data =====
def read_markdown_files(data_dir):
    files = sorted(data_dir.glob("*.md"))
    if not files:
        raise FileNotFoundError(f"No .md files found in {data_dir}")

    documents = []
    for path in files:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": path, "name": path.name, "text": text})
    return documents


def split_into_articles(document):
    text = document["text"]
    source_file = document["name"]
    pattern = re.compile(r"(?m)^###\s+(Điều\s+[^\n]+)")
    matches = list(pattern.finditer(text))

    articles = []
    for idx, match in enumerate(matches):
        start = match.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)
        title = match.group(1).strip()
        body = text[start:end].strip()

        article_hash = hashlib.sha1(f"{source_file}\n{title}\n{body}".encode("utf-8")).hexdigest()[:12]
        article_id = f"{source_file}::{article_hash}"

        articles.append(
            {
                "id": article_id,
                "source_file": source_file,
                "title": title,
                "text": body,
                "char_count": len(body),
            }
        )

    return articles


documents = read_markdown_files(DATA_DIR)
articles = []
for document in documents:
    articles.extend(split_into_articles(document))

if MAX_ARTICLES is not None:
    articles = articles[:MAX_ARTICLES]

print(f"Loaded {len(documents)} markdown files")
print(f"Prepared {len(articles)} legal-article chunks")


Loaded 4 markdown files
Prepared 1407 legal-article chunks


### Extract graph data with cache/resume


In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override = True)

def extract_entities_and_relationships(text):

    client = OpenAI(
        api_key="GROQ_API_KEY",
        base_url="https://api.groq.com/openai/v1"
    )

    prompt = f"""
You are an information extraction system that builds a knowledge graph
from Vietnamese legal or administrative text.

Your task is to extract entities (nodes) and relationships (edges)
that are explicitly mentioned in the text.

Important rules:
- Extract only information that is directly stated in the text.
- Do not infer, guess, summarize, or add external knowledge.
- Keep entity names in Vietnamese exactly as they appear in the text when possible.
- Relationship types must be concise, normalized, and written in Vietnamese.
- Entity types must be concise and written in Vietnamese.
- Merge duplicate entities that refer to the same real-world object.
- Do not extract vague or generic words as entities unless they refer to
  a specific concept, organization, document, person, place, regulation, or event.
- If there are no valid entities or relationships, return empty lists.
- Return valid JSON only.
- Do not include markdown, explanations, comments, or extra text.

Entity extraction guidelines:

Each entity must have:
- "name": the entity name
- "type": the entity category

Suggested entity types:
- "Văn bản pháp luật"
- "Cơ quan"
- "Tổ chức"
- "Cá nhân"
- "Địa điểm"
- "Thời gian"
- "Khái niệm"
- "Quy định"
- "Nghĩa vụ"
- "Quyền lợi"
- "Hành vi"
- "Đối tượng áp dụng"
- "Mức phạt"
- "Điều khoản"

You may create another type if it better represents the entity.

Relationship extraction guidelines:

Each relationship must have:
- "source": the source entity name
- "type": the relationship type
- "target": the target entity name

Suggested relationship types:
- "ban hành"
- "quy định"
- "áp dụng cho"
- "có hiệu lực từ"
- "sửa đổi"
- "bổ sung"
- "thay thế"
- "liên quan đến"
- "có nghĩa vụ"
- "có quyền"
- "bị xử phạt bởi"
- "thuộc"
- "gồm"
- "căn cứ vào"

You may create another relationship type if appropriate.

Output schema:

{{
    "entities": [
        {{
            "name": "...",
            "type": "..."
        }}
    ],
    "relationships": [
        {{
            "source": "...",
            "type": "...",
            "target": "..."
        }}
    ]
}}

Text:
{text}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",

        messages=[
            {
                "role": "system",
                "content": (
                    "You are a precise knowledge graph extraction engine "
                    "for Vietnamese legal and administrative documents. "
                    "Extract only explicitly stated entities and relationships. "
                    "Do not infer, guess, summarize, or add external knowledge. "
                    "Return valid JSON only following the requested schema."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0,

        response_format={
            "type": "json_object"
        },

        # GPT-OSS hỗ trợ low / medium / high
        reasoning_effort="low"
    )

    return json.loads(response.choices[0].message.content)

In [110]:
# ===== Extract entities and relationships with cache/resume =====
def load_extraction_cache(cache_path):
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))
    return {}


def save_extraction_cache(cache_path, cache):
    cache_path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")


def clean_text_value(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()


def normalize_key(value):
    return clean_text_value(value).casefold()


def normalize_extraction(raw_result):
    if isinstance(raw_result, str):
        raw_result = json.loads(raw_result)

    entities = []
    entity_names = set()
    for entity in raw_result.get("entities", []):
        name = clean_text_value(entity.get("name"))
        entity_type = clean_text_value(entity.get("type")) or "Thực thể"
        if not name:
            continue

        key = normalize_key(name)
        if key in entity_names:
            continue
        entity_names.add(key)
        entities.append({"name": name, "type": entity_type})

    relationships = []
    relationship_keys = set()
    for relationship in raw_result.get("relationships", []):
        source = clean_text_value(relationship.get("source"))
        rel_type = clean_text_value(relationship.get("type"))
        target = clean_text_value(relationship.get("target"))
        if not source or not rel_type or not target:
            continue

        key = (normalize_key(source), normalize_key(rel_type), normalize_key(target))
        if key in relationship_keys:
            continue
        relationship_keys.add(key)
        relationships.append({"source": source, "type": rel_type, "target": target})

        for endpoint in (source, target):
            endpoint_key = normalize_key(endpoint)
            if endpoint_key not in entity_names:
                entity_names.add(endpoint_key)
                entities.append({"name": endpoint, "type": "Thực thể"})

    return {"entities": entities, "relationships": relationships}


extraction_cache = load_extraction_cache(CACHE_PATH)

for index, article in enumerate(articles, start=1):
    if article["id"] in extraction_cache:
        continue

    print(f"[{index}/{len(articles)}] Extracting: {article['title']} ({article['source_file']})")
    raw_result = extract_entities_and_relationships(article["text"])
    normalized_result = normalize_extraction(raw_result)

    extraction_cache[article["id"]] = {
        "article": article,
        "entities": normalized_result["entities"],
        "relationships": normalized_result["relationships"],
        "extracted_at": datetime.now().isoformat(timespec="seconds"),
    }
    save_extraction_cache(CACHE_PATH, extraction_cache)
    time.sleep(SLEEP_SECONDS_BETWEEN_CALLS)

print(f"Cached extractions: {len(extraction_cache)}")


[1396/1407] Extracting: Điều 20.5.TT.91.3. Tổ chức thực hiện (de_muc_20_5_an-toan-ve-sinh-lao-dong.md)
[1397/1407] Extracting: Điều 20.5.TT.92.18. Hiệu lực thi hành (de_muc_20_5_an-toan-ve-sinh-lao-dong.md)
[1398/1407] Extracting: Điều 20.5.TT.92.19. Trách nhiệm thi hành (de_muc_20_5_an-toan-ve-sinh-lao-dong.md)
[1399/1407] Extracting: Điều 20.5.TT.93.5. Trách nhiệm thi hành (de_muc_20_5_an-toan-ve-sinh-lao-dong.md)
[1400/1407] Extracting: Điều 20.5.TT.93.6. Tổ chức thực hiện và hiệu lực thi hành (de_muc_20_5_an-toan-ve-sinh-lao-dong.md)
[1401/1407] Extracting: Điều 20.5.TT.94.12. Hiệu lực thi hành (de_muc_20_5_an-toan-ve-sinh-lao-dong.md)
[1402/1407] Extracting: Điều 20.5.TT.95.18. Hiệu lực thi hành (de_muc_20_5_an-toan-ve-sinh-lao-dong.md)
[1403/1407] Extracting: Điều 20.5.TT.96.11. Điều khoản thi hành (de_muc_20_5_an-toan-ve-sinh-lao-dong.md)
[1404/1407] Extracting: Điều 20.5.TT.97.8. Hiệu lực thi hành (de_muc_20_5_an-toan-ve-sinh-lao-dong.md)
[1405/1407] Extracting: Điều 20.5.TT.98

### Aggregate extracted graph


In [111]:
# ===== Aggregate graph =====
def build_graph_payload(cache):
    entities_by_key = {}
    relationships_by_key = {}
    documents_by_name = {}
    articles_by_id = {}

    for record in cache.values():
        article = record["article"]
        source_file = article["source_file"]

        documents_by_name.setdefault(source_file, {"name": source_file})
        articles_by_id[article["id"]] = article

        for entity in record.get("entities", []):
            name = clean_text_value(entity.get("name"))
            entity_type = clean_text_value(entity.get("type")) or "Thực thể"
            if not name:
                continue

            key = normalize_key(name)
            item = entities_by_key.setdefault(
                key,
                {
                    "name": name,
                    "type": entity_type,
                    "mentions": 0,
                    "source_files": set(),
                    "article_ids": set(),
                },
            )
            item["mentions"] += 1
            item["source_files"].add(source_file)
            item["article_ids"].add(article["id"])

        for relationship in record.get("relationships", []):
            source = clean_text_value(relationship.get("source"))
            rel_type = clean_text_value(relationship.get("type"))
            target = clean_text_value(relationship.get("target"))
            if not source or not rel_type or not target:
                continue

            key = (normalize_key(source), normalize_key(rel_type), normalize_key(target))
            item = relationships_by_key.setdefault(
                key,
                {
                    "source": source,
                    "type": rel_type,
                    "target": target,
                    "count": 0,
                    "source_files": set(),
                    "article_ids": set(),
                },
            )
            item["count"] += 1
            item["source_files"].add(source_file)
            item["article_ids"].add(article["id"])

    entities = []
    for entity in entities_by_key.values():
        entity["source_files"] = sorted(entity["source_files"])
        entity["article_ids"] = sorted(entity["article_ids"])
        entities.append(entity)

    relationships = []
    for relationship in relationships_by_key.values():
        relationship["source_files"] = sorted(relationship["source_files"])
        relationship["article_ids"] = sorted(relationship["article_ids"])
        relationships.append(relationship)

    return {
        "documents": sorted(documents_by_name.values(), key=lambda item: item["name"]),
        "articles": sorted(articles_by_id.values(), key=lambda item: item["id"]),
        "entities": sorted(entities, key=lambda item: item["name"]),
        "relationships": sorted(relationships, key=lambda item: (item["source"], item["type"], item["target"])),
    }


graph_payload = build_graph_payload(extraction_cache)
GRAPH_EXPORT_PATH.write_text(json.dumps(graph_payload, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Documents: {len(graph_payload['documents'])}")
print(f"Articles: {len(graph_payload['articles'])}")
print(f"Entities: {len(graph_payload['entities'])}")
print(f"Relationships: {len(graph_payload['relationships'])}")
print(f"Saved graph payload to: {GRAPH_EXPORT_PATH}")


Documents: 4
Articles: 1407
Entities: 6001
Relationships: 8850
Saved graph payload to: /home/nguyen-thanh-dat/Documents/PTHTTM_PTIT/HTTM_Representation/chatbot/kg_graph_all_data.json


### Insert graph into Neo4j


In [112]:
# ===== Insert graph into Neo4j =====
def insert_graph_payload_to_neo4j(kg, payload, reset=False):
    if reset:
        kg.query("""
        MATCH (n)
        WHERE n:KGDocument OR n:KGArticle OR n:KGEntity
        DETACH DELETE n
        """)

    kg.query("CREATE CONSTRAINT kg_document_name IF NOT EXISTS FOR (d:KGDocument) REQUIRE d.name IS UNIQUE")
    kg.query("CREATE CONSTRAINT kg_article_id IF NOT EXISTS FOR (a:KGArticle) REQUIRE a.id IS UNIQUE")
    kg.query("CREATE CONSTRAINT kg_entity_name IF NOT EXISTS FOR (e:KGEntity) REQUIRE e.name IS UNIQUE")

    for document in payload["documents"]:
        kg.query(
            """
            MERGE (d:KGDocument {name: $name})
            SET d.updated_at = datetime()
            """,
            params=document,
        )

    for article in payload["articles"]:
        kg.query(
            """
            MERGE (a:KGArticle {id: $id})
            SET a.title = $title,
                a.source_file = $source_file,
                a.text = $text,
                a.char_count = $char_count,
                a.updated_at = datetime()
            MERGE (d:KGDocument {name: $source_file})
            MERGE (d)-[:CONTAINS]->(a)
            """,
            params=article,
        )

    for entity in payload["entities"]:
        kg.query(
            """
            MERGE (e:KGEntity {name: $name})
            SET e.type = $type,
                e.mentions = $mentions,
                e.source_files = $source_files,
                e.article_ids = $article_ids,
                e.updated_at = datetime()
            """,
            params=entity,
        )

    for record in extraction_cache.values():
        article_id = record["article"]["id"]
        for entity in record.get("entities", []):
            kg.query(
                """
                MATCH (a:KGArticle {id: $article_id})
                MATCH (e:KGEntity {name: $entity_name})
                MERGE (a)-[:MENTIONS]->(e)
                """,
                params={"article_id": article_id, "entity_name": entity["name"]},
            )

    for relationship in payload["relationships"]:
        kg.query(
            """
            MERGE (source:KGEntity {name: $source})
            MERGE (target:KGEntity {name: $target})
            MERGE (source)-[r:RELATES_TO {type: $type}]->(target)
            SET r.count = $count,
                r.source_files = $source_files,
                r.article_ids = $article_ids,
                r.updated_at = datetime()
            """,
            params=relationship,
        )


insert_graph_payload_to_neo4j(kg, graph_payload, reset=RESET_PIPELINE_GRAPH)
print("Inserted knowledge graph into Neo4j")


Inserted knowledge graph into Neo4j


### Visualize graph


In [113]:
# ===== Visualize extracted graph =====
from pyvis.network import Network
from IPython.display import IFrame, display


def visualize_neo4j_kg(kg, output_path=VISUALIZATION_PATH, limit=600):
    rows = kg.query(
        """
        MATCH (a:KGEntity)-[r:RELATES_TO]->(b:KGEntity)
        RETURN
          elementId(a) AS source_id,
          a.name AS source,
          a.type AS source_type,
          elementId(b) AS target_id,
          b.name AS target,
          b.type AS target_type,
          r.type AS rel,
          r.count AS count
        ORDER BY count DESC
        LIMIT $limit
        """,
        params={"limit": limit},
    )

    net = Network(
        height="760px",
        width="100%",
        directed=True,
        notebook=True,
        cdn_resources="in_line",
    )
    net.barnes_hut(gravity=-28000, central_gravity=0.25, spring_length=180, spring_strength=0.04, damping=0.09)

    for row in rows:
        source_title = f"{row['source']}\nType: {row.get('source_type') or 'Thực thể'}"
        target_title = f"{row['target']}\nType: {row.get('target_type') or 'Thực thể'}"
        edge_title = f"{row['rel']}\nCount: {row.get('count') or 1}"

        net.add_node(
            row["source_id"],
            label=row["source"],
            title=source_title,
            group=row.get("source_type") or "Thực thể",
        )
        net.add_node(
            row["target_id"],
            label=row["target"],
            title=target_title,
            group=row.get("target_type") or "Thực thể",
        )
        net.add_edge(
            row["source_id"],
            row["target_id"],
            label=row["rel"],
            title=edge_title,
            value=row.get("count") or 1,
            arrows="to",
        )

    net.write_html(str(output_path))
    display(IFrame(str(output_path), width="100%", height=780))
    return rows


visualized_rows = visualize_neo4j_kg(kg)
print(f"Saved visualization to: {VISUALIZATION_PATH}")


Saved visualization to: /home/nguyen-thanh-dat/Documents/PTHTTM_PTIT/HTTM_Representation/chatbot/kg_visualization_all_data.html


In [115]:
from IPython.display import display
import json


def show_rows(rows, max_rows=20):
    """Display Neo4j query rows nicely in notebook."""
    rows = rows[:max_rows]
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        print(json.dumps(rows, ensure_ascii=False, indent=2))


## Một vài truy vấn xem thử Knowledge Graph


In [124]:
results = kg.query("""
MATCH (entity1:KGEntity)-[r:RELATES_TO]->(entity2:KGEntity)
WHERE entity1.name = 'Bộ Lao động - Thương binh và Xã hội'
  AND r.type = 'quyết định'
RETURN entity2.name AS entity2_name
""")

results

[{'entity2_name': 'thu hồi Giấy phép'},
 {'entity2_name': 'mức chi phí đào tạo nghề, đào tạo ngoại ngữ'}]

In [127]:
# đâu là nơi ban hành quy trình thực hiện việc 
# đánh giá, cấp chứng chỉ kỹ năng nghề nghiệp quốc gia
results = kg.query(
    """MATCH (entity1:KGEntity)-[r:RELATES_TO]->(entity2:KGEntity)
    WHERE r.type = 'ban hành' AND entity2.name = 'quy trình thực hiện việc đánh giá, cấp chứng chỉ kỹ năng nghề quốc gia'
    RETURN entity1.name
    """
)
results

[{'entity1.name': 'Bộ Lao động - Thương binh và Xã hội'}]

### 1. Tổng quan số lượng node và relationship


In [116]:
overview_rows = kg.query("""
MATCH (d:KGDocument)
WITH count(d) AS documents
MATCH (a:KGArticle)
WITH documents, count(a) AS articles
MATCH (e:KGEntity)
WITH documents, articles, count(e) AS entities
MATCH ()-[r:RELATES_TO]->()
RETURN documents, articles, entities, count(r) AS relationships
""")

show_rows(overview_rows)


[
  {
    "documents": 4,
    "articles": 1407,
    "entities": 6267,
    "relationships": 8850
  }
]


### 2. Những thực thể được nhắc đến nhiều nhất


In [117]:
top_entities_rows = kg.query("""
MATCH (a:KGArticle)-[:MENTIONS]->(e:KGEntity)
RETURN
  e.name AS entity,
  e.type AS type,
  count(DISTINCT a) AS article_count,
  coalesce(e.mentions, 0) AS extracted_mentions
ORDER BY article_count DESC, extracted_mentions DESC
LIMIT 25
""")

show_rows(top_entities_rows, max_rows=25)


[
  {
    "entity": "Bộ Lao động - Thương binh và Xã hội",
    "type": "Cơ quan",
    "article_count": 315,
    "extracted_mentions": 317
  },
  {
    "entity": "01/01/2021",
    "type": "Thời gian",
    "article_count": 239,
    "extracted_mentions": 239
  },
  {
    "entity": "Chính phủ",
    "type": "Cơ quan",
    "article_count": 190,
    "extracted_mentions": 190
  },
  {
    "entity": "người lao động",
    "type": "Đối tượng áp dụng",
    "article_count": 181,
    "extracted_mentions": 359
  },
  {
    "entity": "người sử dụng lao động",
    "type": "Đối tượng áp dụng",
    "article_count": 126,
    "extracted_mentions": 305
  },
  {
    "entity": "01/01/2022",
    "type": "Thời gian",
    "article_count": 120,
    "extracted_mentions": 120
  },
  {
    "entity": "01/02/2021",
    "type": "Thực thể",
    "article_count": 111,
    "extracted_mentions": 111
  },
  {
    "entity": "Bộ luật Lao động",
    "type": "Văn bản pháp luật",
    "article_count": 102,
    "extracted_mentions"

### 3. Những loại quan hệ phổ biến nhất


In [118]:
top_relationship_types_rows = kg.query("""
MATCH (:KGEntity)-[r:RELATES_TO]->(:KGEntity)
RETURN
  r.type AS relationship_type,
  count(r) AS edge_count,
  sum(coalesce(r.count, 1)) AS extracted_count
ORDER BY edge_count DESC, extracted_count DESC
LIMIT 25
""")

show_rows(top_relationship_types_rows, max_rows=25)


[
  {
    "relationship_type": "liên quan đến",
    "edge_count": 1742,
    "extracted_count": 1753
  },
  {
    "relationship_type": "có hiệu lực từ",
    "edge_count": 1050,
    "extracted_count": 1588
  },
  {
    "relationship_type": "có nghĩa vụ",
    "edge_count": 711,
    "extracted_count": 753
  },
  {
    "relationship_type": "quy định",
    "edge_count": 652,
    "extracted_count": 668
  },
  {
    "relationship_type": "ban hành",
    "edge_count": 481,
    "extracted_count": 575
  },
  {
    "relationship_type": "áp dụng cho",
    "edge_count": 467,
    "extracted_count": 476
  },
  {
    "relationship_type": "thuộc",
    "edge_count": 372,
    "extracted_count": 394
  },
  {
    "relationship_type": "có quyền",
    "edge_count": 196,
    "extracted_count": 207
  },
  {
    "relationship_type": "căn cứ vào",
    "edge_count": 191,
    "extracted_count": 206
  },
  {
    "relationship_type": "gồm",
    "edge_count": 147,
    "extracted_count": 158
  },
  {
    "relationship_t

### 4. Xem các quan hệ xung quanh một thực thể


In [119]:
ENTITY_NAME = "người lao động"

neighborhood_rows = kg.query("""
MATCH (e:KGEntity)-[r:RELATES_TO]-(other:KGEntity)
WHERE toLower(e.name) CONTAINS toLower($entity_name)
RETURN
  e.name AS center_entity,
  CASE WHEN startNode(r) = e THEN 'outgoing' ELSE 'incoming' END AS direction,
  r.type AS relationship_type,
  other.name AS connected_entity,
  other.type AS connected_type,
  coalesce(r.count, 1) AS count
ORDER BY count DESC, relationship_type, connected_entity
LIMIT 50
""", params={"entity_name": ENTITY_NAME})

show_rows(neighborhood_rows, max_rows=50)


[
  {
    "center_entity": "người lao động",
    "direction": "incoming",
    "relationship_type": "có nghĩa vụ",
    "connected_entity": "người sử dụng lao động",
    "connected_type": "Đối tượng áp dụng",
    "count": 15
  },
  {
    "center_entity": "Người lao động",
    "direction": "incoming",
    "relationship_type": "áp dụng cho",
    "connected_entity": "Người sử dụng lao động",
    "connected_type": null,
    "count": 4
  },
  {
    "center_entity": "người lao động",
    "direction": "incoming",
    "relationship_type": "có quyền",
    "connected_entity": "người sử dụng lao động",
    "connected_type": "Đối tượng áp dụng",
    "count": 3
  },
  {
    "center_entity": "người lao động",
    "direction": "incoming",
    "relationship_type": "gửi",
    "connected_entity": "Trung tâm dịch vụ việc làm",
    "connected_type": "Tổ chức",
    "count": 3
  },
  {
    "center_entity": "người lao động",
    "direction": "outgoing",
    "relationship_type": "liên quan đến",
    "connected_

### 5. Truy ngược các điều luật/văn bản nhắc tới một thực thể


In [120]:
ENTITY_NAME = "người lao động"

source_articles_rows = kg.query("""
MATCH (d:KGDocument)-[:CONTAINS]->(a:KGArticle)-[:MENTIONS]->(e:KGEntity)
WHERE toLower(e.name) CONTAINS toLower($entity_name)
RETURN
  e.name AS entity,
  d.name AS source_file,
  a.title AS article_title,
  left(a.text, 450) AS text_preview
ORDER BY source_file, article_title
LIMIT 20
""", params={"entity_name": ENTITY_NAME})

show_rows(source_articles_rows, max_rows=20)


[
  {
    "entity": "tổ chức đại diện người lao động tại cơ sở",
    "source_file": "de_muc_20_2_lao-dong.md",
    "article_title": "Điều 20.2.LQ.1. Phạm vi điều chỉnh",
    "text_preview": "### Điều 20.2.LQ.1. Phạm vi điều chỉnh\n\n(Điều 1 Bộ luật số 45/2019/QH14 Bộ Luật lao động ngày 20/11/2019 của Quốc hội, có hiệu lực thi hành kể từ ngày 01/01/2021 )\nBộ luật Lao động quy định tiêu chuẩn lao động; quyền, nghĩa vụ, trách nhiệm của người lao động, người sử dụng lao động, tổ chức đại diện người lao động tại cơ sở, tổ chức đại diện người sử dụng lao động trong quan hệ lao động và các quan hệ khác liên quan trực tiếp đến quan hệ lao động"
  },
  {
    "entity": "người lao động",
    "source_file": "de_muc_20_2_lao-dong.md",
    "article_title": "Điều 20.2.LQ.1. Phạm vi điều chỉnh",
    "text_preview": "### Điều 20.2.LQ.1. Phạm vi điều chỉnh\n\n(Điều 1 Bộ luật số 45/2019/QH14 Bộ Luật lao động ngày 20/11/2019 của Quốc hội, có hiệu lực thi hành kể từ ngày 01/01/2021 )\nBộ luật Lao động quy

### 6. Tìm đường liên kết ngắn giữa hai thực thể


In [121]:
SOURCE_ENTITY = "người lao động"
TARGET_ENTITY = "người sử dụng lao động"

path_rows = kg.query("""
MATCH (source:KGEntity), (target:KGEntity)
WHERE toLower(source.name) CONTAINS toLower($source_entity)
  AND toLower(target.name) CONTAINS toLower($target_entity)
  AND source <> target
MATCH path = shortestPath((source)-[:RELATES_TO*..4]-(target))
RETURN
  source.name AS source,
  target.name AS target,
  [node IN nodes(path) | node.name] AS entity_path,
  [rel IN relationships(path) | rel.type] AS relationship_path,
  length(path) AS hops
ORDER BY hops
LIMIT 10
""", params={"source_entity": SOURCE_ENTITY, "target_entity": TARGET_ENTITY})

show_rows(path_rows, max_rows=10)


[
  {
    "source": "báo cáo về việc quản lý sức khỏe người lao động",
    "target": "Người sử dụng lao động",
    "entity_path": [
      "báo cáo về việc quản lý sức khỏe người lao động",
      "Người sử dụng lao động"
    ],
    "relationship_path": [
      "có nghĩa vụ"
    ],
    "hops": 1
  },
  {
    "source": "bảo đảm không ngược đãi người lao động",
    "target": "Người sử dụng lao động",
    "entity_path": [
      "bảo đảm không ngược đãi người lao động",
      "Người sử dụng lao động"
    ],
    "relationship_path": [
      "có nghĩa vụ"
    ],
    "hops": 1
  },
  {
    "source": "Tổ chức đại diện người lao động",
    "target": "người sử dụng lao động",
    "entity_path": [
      "Tổ chức đại diện người lao động",
      "người sử dụng lao động"
    ],
    "relationship_path": [
      "có nghĩa vụ"
    ],
    "hops": 1
  },
  {
    "source": "Yêu cầu tổ chức đại diện người lao động thương lượng",
    "target": "Người sử dụng lao động",
    "entity_path": [
      "Yêu cầu tổ c

In [139]:
from dotenv import load_dotenv
import os

import textwrap

# Langchain
from langchain_neo4j import Neo4jGraph
from langchain_neo4j import Neo4jVector
from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_core.prompts import PromptTemplate
from langchain_neo4j import GraphCypherQAChain
from langchain_openai import ChatOpenAI
# Warning control
import warnings
warnings.filterwarnings("ignore")
load_dotenv(override = True)

True

In [146]:
CYPHER_GENERATION_TEMPLATE = """
You are an expert Neo4j Cypher generator for a Vietnamese labor-law knowledge graph.

Task: Generate a Cypher query that answers the user's question.

Graph schema:
- (:KGDocument {{name}})
- (:KGArticle {{id, title, text}})
- (:KGEntity {{name}})
- (:KGDocument)-[:CONTAINS]->(:KGArticle)
- (:KGArticle)-[:MENTIONS]->(:KGEntity)
- (:KGEntity)-[:RELATES_TO {{type, source_files, article_ids}}]->(:KGEntity)

Important rules:
- Use only the schema above. Do not invent labels, relationship types, or properties.
- NEVER use the label :Entity. The entity label is always :KGEntity.
- NEVER put Vietnamese semantic relationship names inside square brackets, such as [r:có nghĩa vụ] or [r:quyết định]. This is invalid Cypher for this graph.
- The only relationship type between entities is :RELATES_TO.
- The semantic relationship name is stored in r.type, not in the Neo4j relationship type.
- Always use [r:RELATES_TO] when querying semantic relationships between entities.
- To query 'có nghĩa vụ', write [r:RELATES_TO] and WHERE r.type = 'có nghĩa vụ'.
- To query 'có quyền', write [r:RELATES_TO] and WHERE r.type = 'có quyền'.
- To query 'quyết định', write [r:RELATES_TO] and WHERE r.type = 'quyết định'.
- For Vietnamese entity names, prefer case-insensitive matching with toLower(... ) CONTAINS toLower(...).
- Return clear aliases such as source, relationship_type, target, article_title, article_text.
- Add LIMIT 20 unless the question explicitly asks for counts or aggregation.
- Return only the Cypher query. Do not include explanations, markdown, or backticks.

Invalid patterns and their correct replacements:
- WRONG: MATCH (a:Entity)-[r:có nghĩa vụ]->(b:Entity)
- RIGHT: MATCH (a:KGEntity)-[r:RELATES_TO]->(b:KGEntity) WHERE r.type = 'có nghĩa vụ'
- WRONG: MATCH (a:KGEntity)-[r:quyết định]->(b:KGEntity)
- RIGHT: MATCH (a:KGEntity)-[r:RELATES_TO]->(b:KGEntity) WHERE r.type = 'quyết định'

Few-shot examples:

Question: Bộ Lao động - Thương binh và Xã hội quyết định những gì?
Cypher:
MATCH (entity1:KGEntity)-[r:RELATES_TO]->(entity2:KGEntity)
WHERE toLower(entity1.name) CONTAINS toLower('Bộ Lao động - Thương binh và Xã hội')
  AND r.type = 'quyết định'
RETURN entity1.name AS source, r.type AS relationship_type, entity2.name AS target
LIMIT 20

Question: Các thực thể nào có nghĩa vụ với người lao động?
Cypher:
MATCH (entity1:KGEntity)-[r:RELATES_TO]->(entity2:KGEntity)
WHERE r.type = 'có nghĩa vụ'
  AND toLower(entity2.name) CONTAINS toLower('người lao động')
RETURN entity1.name AS source, r.type AS relationship_type, entity2.name AS target
LIMIT 20

Question: Người sử dụng lao động có những quyền gì?
Cypher:
MATCH (entity1:KGEntity)-[r:RELATES_TO]->(entity2:KGEntity)
WHERE toLower(entity1.name) CONTAINS toLower('người sử dụng lao động')
  AND r.type = 'có quyền'
RETURN entity1.name AS source, r.type AS relationship_type, entity2.name AS target
LIMIT 20

Question: Người sử dụng lao động có nghĩa vụ gì?
Cypher:
MATCH (entity1:KGEntity)-[r:RELATES_TO]->(entity2:KGEntity)
WHERE toLower(entity1.name) CONTAINS toLower('người sử dụng lao động')
  AND r.type = 'có nghĩa vụ'
RETURN entity1.name AS source, r.type AS relationship_type, entity2.name AS target
LIMIT 20

Question: Những bài viết nào nhắc đến tổ chức đại diện người lao động?
Cypher:
MATCH (article:KGArticle)-[:MENTIONS]->(entity:KGEntity)
WHERE toLower(entity.name) CONTAINS toLower('tổ chức đại diện người lao động')
RETURN article.id AS article_id, article.title AS article_title, entity.name AS entity
LIMIT 20

Question: Có bao nhiêu loại quan hệ trong graph?
Cypher:
MATCH (:KGEntity)-[r:RELATES_TO]->(:KGEntity)
RETURN count(DISTINCT r.type) AS relationship_type_count

Question: Liệt kê các loại quan hệ phổ biến nhất.
Cypher:
MATCH (:KGEntity)-[r:RELATES_TO]->(:KGEntity)
RETURN r.type AS relationship_type, count(*) AS count
ORDER BY count DESC, relationship_type
LIMIT 20

Question: Tìm đường liên kết ngắn giữa người lao động và người sử dụng lao động.
Cypher:
MATCH (source:KGEntity), (target:KGEntity)
WHERE toLower(source.name) CONTAINS toLower('người lao động')
  AND toLower(target.name) CONTAINS toLower('người sử dụng lao động')
  AND source <> target
MATCH path = shortestPath((source)-[:RELATES_TO*..4]-(target))
RETURN source.name AS source,
       target.name AS target,
       [node IN nodes(path) | node.name] AS entity_path,
       [rel IN relationships(path) | rel.type] AS relationship_path,
       length(path) AS hops
ORDER BY hops
LIMIT 10

Auto-generated schema from the graph database, for reference only. If it conflicts with the rules above, follow the rules above:
{schema}

Question:
{question}

Cypher:
"""

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template=CYPHER_GENERATION_TEMPLATE,
)


In [147]:
cypherChain = GraphCypherQAChain.from_llm(
    ChatOpenAI(
        temperature=0,
        openai_api_key=os.getenv("OPENAI_API_KEY")
        ),
    graph=kg,
    verbose=True,
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    allow_dangerous_requests=True
)

In [148]:
def prettyCypherChain(question: str) -> str:
    result = cypherChain.invoke({"query": question})
    response = result.get("result", result)
    print("---result", result)
    print(textwrap.fill(str(response), 60))
    return response

In [149]:
prettyCypherChain("Người sử dụng lao động có nghĩa vụ gì?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (entity1:KGEntity)-[r:RELATES_TO]->(entity2:KGEntity)
WHERE toLower(entity1.name) CONTAINS toLower('người sử dụng lao động')
  AND r.type = 'có nghĩa vụ'
RETURN entity1.name AS source, r.type AS relationship_type, entity2.name AS target
LIMIT 20
Full Context:
[{'source': 'người sử dụng lao động', 'relationship_type': 'có nghĩa vụ', 'target': 'Ủy ban nhân dân cấp tỉnh'}, {'source': 'người sử dụng lao động', 'relationship_type': 'có nghĩa vụ', 'target': 'Ủy ban nhân dân cấp huyện'}, {'source': 'người sử dụng lao động', 'relationship_type': 'có nghĩa vụ', 'target': 'đăng ký, khám, kiểm tra sức khỏe nghĩa vụ quân sự'}, {'source': 'người sử dụng lao động', 'relationship_type': 'có nghĩa vụ', 'target': 'đóng bảo hiểm thất nghiệp'}, {'source': 'người sử dụng lao động', 'relationship_type': 'có nghĩa vụ', 'target': 'đánh giá nguy cơ rủi ro về an toàn, vệ sinh lao động'}, {'source': 'người sử dụng lao động', 'relationship_type

'Người sử dụng lao động có nghĩa vụ đăng ký, khám, kiểm tra sức khỏe nghĩa vụ quân sự, đóng bảo hiểm thất nghiệp, đánh giá nguy cơ rủi ro về an toàn, vệ sinh lao động, xây dựng kế hoạch ứng cứu khẩn cấp, tổ chức đại diện người lao động tại cơ sở, tổ chức thực hiện công tác an toàn, vệ sinh lao động, tổ chức khám sức khỏe cho người lao động, tổ chức huấn luyện sơ cứu, cấp cứu cho người lao động.'